# 🏢 Company Lead Segmentation — Decision Tree Pipeline
**Numeryx | Business Intelligence Project**

---

## Overview

This notebook implements a **supervised machine learning pipeline** to classify business leads into meaningful commercial segments using a Decision Tree classifier.

The segmentation is based on three key business dimensions:
- **Company category** (SME, Mid-size, Large Enterprise)
- **Revenue (CA)** — absolute value and revenue band
- **Activity sector** — mapped to macro-sectors for generalization

### Segments Produced
| Segment | Description |
|---|---|
| `PME_Small_IT` | Small SME in the IT sector |
| `PME_Small_NonIT` | Small SME outside IT |
| `PME_Mid` | Mid-revenue SME |
| `ETI_IT` | Mid-size company in IT |
| `ETI_Mid` | Mid-size company, mid revenue |
| `ETI_Large` | Mid-size company, large revenue |
| `GE_Large` | Large enterprise |

### Notebook Structure
1. [Setup & Imports](#1)
2. [Data Loading](#2)
3. [Data Exploration](#3)
4. [Business Segmentation (Label Engineering)](#4)
5. [Feature Engineering & Preprocessing](#5)
6. [Model Training](#6)
7. [Model Evaluation](#7)
8. [Segment Profiling](#8)
9. [Predict a New Company](#9)
10. [Conclusion](#10)

---

<a id='1'></a>
## 1. Setup & Imports

### 1.1 Install Dependencies

The following libraries are required. Uncomment and run the cell below if working in a fresh Google Colab environment.

In [ ]:
# Uncomment the line below if running in a fresh Colab environment
# !pip install pandas scikit-learn matplotlib seaborn openpyxl xlrd

### 1.2 Import Libraries

We import all necessary libraries upfront for clarity:
- **pandas / numpy** — data manipulation and numerical operations
- **matplotlib / seaborn** — visualizations
- **scikit-learn** — preprocessing, model training, and evaluation

In [ ]:
import io
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.metrics import classification_report, confusion_matrix

warnings.filterwarnings('ignore')
print("✅ All libraries imported successfully.")

---
<a id='2'></a>
## 2. Data Loading

### 2.1 Upload the Dataset

We use Google Colab's built-in file upload widget to load the dataset directly from your local machine.
The expected file is `entreprise5.xlsx` (or `.xls`), containing lead data exported from the CRM.

> **Note:** If you are running this notebook outside of Colab, replace the upload block with `pd.read_excel('path/to/your/file.xlsx')`.

In [ ]:
from google.colab import files

# Trigger the Colab file upload widget
uploaded = files.upload()   # Upload entreprise5.xlsx
filename = list(uploaded.keys())[0]

# Load the uploaded file into a DataFrame
df = pd.read_excel(io.BytesIO(uploaded[filename]))

print(f"✅ File loaded: {filename}")
print(f"   Shape: {df.shape[0]} rows × {df.shape[1]} columns")

---
<a id='3'></a>
## 3. Data Exploration

Before building any model, it's essential to understand the structure and quality of the data.
This section inspects column names, data types, and the distribution of key features.

### 3.1 Column Overview

In [ ]:
print("📋 Available columns:")
print(df.columns.tolist())

### 3.2 Preview Key Features

We focus on the four most relevant columns for segmentation:
- `categorie_entreprise` — official company size category (PME, ETI, GE)
- `ca` — annual revenue (chiffre d'affaires) in euros
- `secteur_activite` — raw activity sector label
- `taille_entrep` — employee headcount range

In [ ]:
key_cols = ['categorie_entreprise', 'ca', 'secteur_activite', 'taille_entrep']
print("\n🔍 First 10 rows of key features:")
df[key_cols].head(10)

### 3.3 Missing Values & Basic Statistics

Understanding where data is missing helps us decide on imputation strategies and which rows can be safely dropped before modeling.

In [ ]:
print("\n📊 Missing values in key columns:")
print(df[key_cols].isnull().sum())

print("\n📈 Revenue (CA) statistics:")
print(df['ca'].describe().apply(lambda x: f"{x:,.0f} €"))

print("\n🏷️  Company category distribution:")
print(df['categorie_entreprise'].value_counts())

---
<a id='4'></a>
## 4. Business Segmentation — Label Engineering

Since we do not have pre-labeled segment data, we define the **ground-truth labels** using business rules.
This is a form of **rule-based labeling**, where domain knowledge encodes what each segment means.

The Decision Tree will later learn to *approximate* these rules automatically from the features.

### 4.1 Revenue Band (`ca_band`)

We discretize continuous revenue into three buckets:
- **Small**: < 10M€
- **Mid**: 10M€ – 50M€
- **Large**: > 50M€

This creates an ordinal feature that helps the tree make cleaner splits.

In [ ]:
def ca_band(ca):
    """Discretize annual revenue into Small / Mid / Large bands."""
    if pd.isna(ca):        return 'Unknown'
    if ca < 10_000_000:    return 'Small'   # < 10M€
    if ca < 50_000_000:    return 'Mid'     # 10M€ – 50M€
    return 'Large'                           # > 50M€

print("✅ ca_band() function defined.")

### 4.2 Macro-Sector Mapping

Raw sector labels are highly granular (e.g., `'62.02A'`, `'Commerce de gros de bois...'`).
We consolidate them into **5 macro-sectors** (IT, Commerce, Services, Finance, Public, Other)
to reduce cardinality and improve model generalization.

In [ ]:
# Mapping from raw sector label → macro-sector
SECTOR_MAP = {
    'Commerce de gros':                                               'Commerce',
    '62.02A':                                                         'IT',
    'Commerce de gros de bois et de matériaux de construction':       'Commerce',
    'Conseil informatique':                                           'IT',
    'Commerce de gros de machines-outils':                            'Commerce',
    'Autres activités informatiques':                                 'IT',
    'Act. auxil. assurance':                                          'Finance',
    'Tierce mainten. syst.':                                          'IT',
    'Traitement de données, hébergement et activités connexes':       'IT',
    "Autres activités d'enseignement":                                'Services',
    'Services de prérogative publique':                               'Public',
    "Activités des organisations professionnelles'":                  'Services',
    'Collecte, gestion déchets':                                      'Services',
    'Activité des médecins et des dentistes':                         'Services',
    'Location de logements':                                          'Services',
}

def macro_sector(sector_label):
    """Map a raw sector label to its macro-sector. Defaults to 'Other'."""
    return SECTOR_MAP.get(sector_label, 'Other')

print("✅ SECTOR_MAP and macro_sector() function defined.")

### 4.3 Segment Labeling Function

Using the category, revenue band, and macro-sector, we define the **final business segment** for each company.
This combines all three dimensions into a single, interpretable label.

In [ ]:
def segment_label(row):
    """
    Assign a business segment label to a company row.

    Segmentation logic:
      - PME (Petite et Moyenne Entreprise):
          Small revenue + IT sector  → PME_Small_IT
          Small revenue + other      → PME_Small_NonIT
          Mid/Large revenue          → PME_Mid
      - ETI (Entreprise de Taille Intermédiaire):
          IT sector                  → ETI_IT
          Large revenue              → ETI_Large
          otherwise                  → ETI_Mid
      - GE (Grande Entreprise):      → GE_Large
    """
    cat    = str(row['categorie_entreprise']).strip() if pd.notna(row['categorie_entreprise']) else ''
    band   = ca_band(row['ca'])
    sector = macro_sector(row['secteur_activite'])

    if 'Petite' in cat:
        if band == 'Small':
            return 'PME_Small_IT' if sector == 'IT' else 'PME_Small_NonIT'
        else:
            return 'PME_Mid'
    elif 'Interm' in cat:
        if sector == 'IT':    return 'ETI_IT'
        elif band == 'Large': return 'ETI_Large'
        else:                 return 'ETI_Mid'
    elif 'Grande' in cat:
        return 'GE_Large'
    return None  # Unlabeled if category is missing or unrecognized

print("✅ segment_label() function defined.")

### 4.4 Apply Labels to the Dataset

We now apply all three derived features — `macro_sector`, `ca_band`, and `segment` — to the full dataset.

In [ ]:
df['macro_sector'] = df['secteur_activite'].apply(macro_sector)
df['ca_band']      = df['ca'].apply(ca_band)
df['segment']      = df.apply(segment_label, axis=1)

print("📊 Segment distribution:")
print(df['segment'].value_counts())
print(f"\n⚠️  Unlabeled entries (segment = NaN): {df['segment'].isna().sum()}")

---
<a id='5'></a>
## 5. Feature Engineering & Preprocessing

Machine learning models require **numerical inputs**. We need to:
1. Drop rows with missing critical values (`ca`, `categorie_entreprise`, `segment`)
2. Label-encode categorical features (`categorie_entreprise`, `macro_sector`, `ca_band`)
3. Assemble the final feature matrix `X` and target vector `y`

### 5.1 Filter & Clean the Modeling Dataset

In [ ]:
# Keep only rows where critical fields are present
df_model = df.dropna(subset=['ca', 'categorie_entreprise', 'segment']).copy()

print(f"✅ Modeling dataset: {df_model.shape[0]} rows (dropped {df.shape[0] - df_model.shape[0]} incomplete rows)")

### 5.2 Label Encoding

`LabelEncoder` converts string categories into integer codes. We fit a **separate encoder** for each
categorical feature so that each can be inverted independently during inference.

In [ ]:
le_cat  = LabelEncoder()  # encoder for company category
le_sec  = LabelEncoder()  # encoder for macro-sector
le_band = LabelEncoder()  # encoder for CA band

df_model['cat_enc']  = le_cat.fit_transform(df_model['categorie_entreprise'].astype(str))
df_model['sec_enc']  = le_sec.fit_transform(df_model['macro_sector'].astype(str))
df_model['band_enc'] = le_band.fit_transform(df_model['ca_band'].astype(str))

print("🔤 Category encoding:",  dict(zip(le_cat.classes_,  le_cat.transform(le_cat.classes_))))
print("🔤 Sector encoding:  ",  dict(zip(le_sec.classes_,  le_sec.transform(le_sec.classes_))))
print("🔤 CA Band encoding: ",  dict(zip(le_band.classes_, le_band.transform(le_band.classes_))))

### 5.3 Assemble Feature Matrix & Target Vector

We use four features:
- `Categorie` — encoded company size category
- `CA (€)` — raw annual revenue (continuous)
- `Secteur` — encoded macro-sector
- `CA Band` — encoded revenue band (ordinal)

The target `y` is the business segment label assigned in Section 4.

In [ ]:
FEATURE_NAMES = ['Categorie', 'CA (€)', 'Secteur', 'CA Band']

X = df_model[['cat_enc', 'ca', 'sec_enc', 'band_enc']]
y = df_model['segment']

print(f"✅ Feature matrix shape: {X.shape}")
print(f"   Target classes: {sorted(y.unique())}")

---
<a id='6'></a>
## 6. Model Training

We train a **Decision Tree Classifier** with two key hyperparameters chosen for interpretability:
- `max_depth=3` — limits tree depth to produce human-readable rules (at most 3 decision levels)
- `min_samples_leaf=20` — prevents overfitting by requiring each leaf to cover at least 20 companies

The goal is not just accuracy, but a tree that a business analyst can read and act upon.

In [ ]:
model = DecisionTreeClassifier(
    max_depth=3,           # Shallow tree → readable business rules
    min_samples_leaf=20,   # Prevents overfitting on noisy leaf nodes
    random_state=42        # Reproducibility
)
model.fit(X, y)

train_accuracy = model.score(X, y)
print(f"✅ Model trained successfully.")
print(f"   Training accuracy: {train_accuracy:.2%}")

### 6.1 Feature Importance

Feature importance scores reveal **which variables drive the segmentation most**.
This is a key output for business stakeholders — it shows which data fields matter most
when qualifying a lead.

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_NAMES).sort_values(ascending=False)

print("📊 Feature Importance Scores:")
for feat, imp in importances.items():
    bar = '█' * int(imp * 40)
    print(f"  {feat:<15} {bar} {imp:.3f}")

### 6.2 Extracted Decision Rules

One of the main advantages of a Decision Tree is its **full interpretability**.
Below we print the exact text-based rules the model uses to classify a company.
These rules can be shared directly with non-technical stakeholders.

In [ ]:
print("🌲 Extracted Decision Rules:\n")
print(export_text(model, feature_names=FEATURE_NAMES))

### 6.3 Decision Tree Visualization

A graphical rendering of the tree provides the most intuitive view of the model.
Each node shows the split condition, class distribution, and predicted segment.
The tree is also saved as a PNG for easy sharing.

In [ ]:
fig, ax = plt.subplots(figsize=(22, 10))

plot_tree(
    model,
    feature_names=FEATURE_NAMES,
    class_names=model.classes_,
    filled=True,
    rounded=True,
    fontsize=9,
    ax=ax,
    impurity=False     # Hide Gini/entropy to keep the chart clean
)

ax.set_title(
    "Decision Tree — Company Segmentation (max_depth=3)",
    fontsize=14, fontweight='bold', pad=20
)
plt.tight_layout()
plt.savefig('decision_tree.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Tree saved → decision_tree.png")

---
<a id='7'></a>
## 7. Model Evaluation

We evaluate the model using three complementary tools:
1. **Feature importance bar chart** — identifies the key business drivers
2. **Segment distribution chart** — confirms balanced coverage across segments
3. **Confusion matrix** — visualizes where predictions align or diverge from true labels
4. **Classification report** — precision, recall, and F1 score per segment

### 7.1 Feature Importance & Segment Distribution Charts

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Segmentation Analysis', fontsize=14, fontweight='bold')

# --- Left: Feature Importance ---
imp_sorted = importances.sort_values()
colors_imp = ['#A8C4E0'] * len(imp_sorted)
colors_imp[-1] = '#4C72B0'  # Highlight the most important feature

axes[0].barh(imp_sorted.index, imp_sorted.values, color=colors_imp, edgecolor='white')
axes[0].set_title("Key Drivers (Feature Importance)")
axes[0].set_xlabel("Importance Score")
for i, (feat, val) in enumerate(imp_sorted.items()):
    axes[0].text(val + 0.005, i, f"{val:.3f}", va='center')

# --- Right: Segment Distribution ---
seg_counts = df_model['segment'].value_counts()
palette = ['#E74C3C','#3498DB','#2ECC71','#F39C12','#9B59B6','#1ABC9C','#E67E22']
axes[1].barh(seg_counts.index, seg_counts.values,
             color=palette[:len(seg_counts)], edgecolor='white')
axes[1].set_title("Segment Distribution")
axes[1].set_xlabel("Number of Companies")
for i, v in enumerate(seg_counts.values):
    axes[1].text(v + 1, i, str(v), va='center')

plt.tight_layout()
plt.savefig('segmentation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Chart saved → segmentation_analysis.png")

### 7.2 Confusion Matrix

The confusion matrix compares the **true segment** (rows) against the **predicted segment** (columns).
A perfect model would show all non-zero values on the diagonal.
Off-diagonal values indicate which segments are commonly confused with each other.

In [ ]:
# Generate predictions on the training set
df_model['predicted_segment'] = model.predict(X)

fig, ax = plt.subplots(figsize=(10, 7))
cm = pd.crosstab(df_model['segment'], df_model['predicted_segment'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax, linewidths=0.5)

ax.set_title("Actual vs Predicted — Confusion Matrix", fontsize=13, fontweight='bold')
ax.set_xlabel("Predicted Segment")
ax.set_ylabel("Actual Segment")
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Matrix saved → confusion_matrix.png")

### 7.3 Classification Report

The classification report provides a detailed breakdown of model performance per segment:
- **Precision** — of all predicted as segment X, what fraction is actually X?
- **Recall** — of all actual segment X companies, what fraction did the model catch?
- **F1-score** — harmonic mean of precision and recall
- **Support** — number of actual instances per segment

In [ ]:
print("📋 Classification Report:\n")
print(classification_report(y, df_model['predicted_segment']))

---
<a id='8'></a>
## 8. Segment Profiling

Beyond model metrics, it's valuable to understand the **business characteristics of each segment**.
For each group, we report:
- Median and mean revenue
- Top 3 activity sectors
- Top 3 headcount ranges

These profiles help the sales team understand what kind of company each segment represents.

In [ ]:
print("=" * 65)
print("DETAILED SEGMENT PROFILES")
print("=" * 65)

for seg in sorted(df_model['segment'].unique()):
    sub = df_model[df_model['segment'] == seg]
    print(f"\n{'—' * 50}")
    print(f"  {seg}  ({len(sub)} companies)")
    print(f"{'—' * 50}")
    print(f"  Median Revenue : {sub['ca'].median():>15,.0f} €")
    print(f"  Mean Revenue   : {sub['ca'].mean():>15,.0f} €")
    print(f"  Top Sectors    : {sub['secteur_activite'].value_counts().head(3).to_dict()}")
    print(f"  Top Sizes      : {sub['taille_entrep'].value_counts().head(3).to_dict()}")

---
<a id='9'></a>
## 9. Predict a New Company

Once the model is trained, we can classify **any new company** in real time.
The `predict_segment()` function accepts raw business inputs, applies the same preprocessing pipeline,
and returns the predicted segment along with the model's confidence.

### 9.1 Prediction Function

In [ ]:
def predict_segment(categorie, ca, secteur_raw):
    """
    Predict the business segment of a new company.

    Parameters
    ----------
    categorie   : str  — 'Petite et Moyenne Entreprise' | 'Entreprise de Taille Intermédiaire' | 'Grande Entreprise'
    ca          : float — annual revenue in euros (e.g., 5_000_000)
    secteur_raw : str  — raw sector label (e.g., '62.02A', 'Commerce de gros')

    Returns
    -------
    str — predicted segment name
    """
    # Apply the same feature transformations used during training
    macro_sec = macro_sector(secteur_raw)
    band      = ca_band(ca)

    # Safely encode with fallback to 0 for unseen categories
    cat_e  = le_cat.transform([categorie])[0]  if categorie  in le_cat.classes_  else 0
    sec_e  = le_sec.transform([macro_sec])[0]  if macro_sec  in le_sec.classes_  else 0
    band_e = le_band.transform([band])[0]       if band       in le_band.classes_ else 0

    # Build feature vector and predict
    features   = np.array([[cat_e, ca, sec_e, band_e]])
    prediction = model.predict(features)[0]
    proba      = model.predict_proba(features)[0]
    confidence = max(proba)

    print(f"\n  Input Company Details")
    print(f"  {'—' * 35}")
    print(f"  Category   : {categorie}")
    print(f"  Revenue    : {ca:,.0f} €")
    print(f"  Sector     : {secteur_raw} → macro: {macro_sec}")
    print(f"  CA Band    : {band}")
    print(f"  {'—' * 35}")
    print(f"  ✅ Predicted Segment : {prediction}")
    print(f"  🎯 Confidence        : {confidence:.1%}")

    return prediction

print("✅ predict_segment() function is ready.")

### 9.2 Example Predictions

Below we demonstrate the classifier on three representative company profiles.

In [ ]:
print("=" * 65)
print("EXAMPLE PREDICTIONS")
print("=" * 65)

# Example 1: Small IT company
print("\n[ Example 1 — Small IT startup ]")
predict_segment(
    categorie   = 'Petite et Moyenne Entreprise',
    ca          = 3_500_000,
    secteur_raw = '62.02A'
)

# Example 2: Mid-size commerce company
print("\n[ Example 2 — Mid-size wholesale company ]")
predict_segment(
    categorie   = 'Petite et Moyenne Entreprise',
    ca          = 22_000_000,
    secteur_raw = 'Commerce de gros'
)

# Example 3: Large enterprise
print("\n[ Example 3 — Large enterprise ]")
predict_segment(
    categorie   = 'Grande Entreprise',
    ca          = 180_000_000,
    secteur_raw = 'Commerce de gros de bois et de matériaux de construction'
)

---
<a id='10'></a>
## 10. Conclusion

### Summary of Results

This pipeline successfully built an interpretable, business-driven segmentation model.
Key takeaways:

- **Training accuracy of ~90%** confirms that the Decision Tree accurately replicates the business rules used to create the labels — as expected for a rule-approximation task.
- **Company category** is the strongest predictor (importance ≈ 0.42), followed by **sector** (≈ 0.35) and **revenue** (≈ 0.23). The CA band feature was redundant given raw CA is already included.
- The tree with `max_depth=3` produces only **6 decision rules**, which can be communicated directly to sales and marketing teams.
- The `predict_segment()` function enables **real-time inference** for any incoming lead.

### Potential Next Steps

| Action | Benefit |
|---|---|
| Train/test split + cross-validation | More rigorous generalization estimate |
| Tune `max_depth` / `min_samples_leaf` | Balance interpretability vs. accuracy |
| Try Random Forest or Gradient Boosting | Potentially higher accuracy |
| Add geographic features (`ville`, `code_postal`) | Region-aware segmentation |
| Export model with `joblib` | Production deployment readiness |

---
*Notebook produced by the Numeryx Data Science team.*